# R1-03 Dashboard prep
Prepares everything for the Tableau dashboard except the `.twbx` itself, which Ayhan builds in Tableau.

Creates:
- `dashboard/data/*.csv`: seven small Tableau-ready extracts, one per metric, computed at run time from `data/processed` (M7 from `langsmith/trace_summary.csv`). Each has a `source` and a `synthetic` column.
- `dashboard/data/dashboard_all.csv`: all 7 metrics stacked in one table with fixed columns, for Tableau.
- `dashboard/dashboard_documentation.md`: headlines, chart suggestions, sources, build steps, the Tableau Public link, the screenshots, the workbook name, the 2-minute test result and the checklist.

**Small extension of the spec (needed to build the .twbx):** the spec names only the workbook and the documentation. The CSV extracts are added so Tableau has clean inputs and every number stays traceable.

Tags: A problem (M1, M2, M3), B use case (M4 triage, M5 team view UC3), C trust (M6 pulse, M7 LangSmith trace evidence).

**SLA gap (M3):** the Spreekorb-today numbers are synthetic. Two extra rows, agent touch time per email (manual today and with the AI draft), are read at run time from `cost_estimation/cost_analysis.md` (R1-06): manual is an estimate (Ayhan), AI-assisted is a self-reported average (Ayhan). Touch time is minutes, customer wait is hours.

**Pulse (M6):** computed from `synthetic_team_pulse.csv` (80 made-up rows, team level, no id), not from the summary file, because the summary shows almost every cell as `<5`. Any overall count under 5 is still shown as `<5`. The headline values are computed, never typed.

**Tobi-Bueck (M4):** only `tobi_tuning_150.csv` is read, never the test file. It is labelled synthetic=True (confirmed synthetic by Ayhan, 2026-11-26).

**Minimum group (M5):** the rule of at least 5 distinct agents per team-month was asserted in R1-02 before agent names were dropped and saved as the `group_ok` flag. This card relies on that flag. M5 shows CSAT per team only.

**Suppression (M2, M5):** groups with n under 30 stay in the CSV with n shown, the value blank and `shown = False`.

**M7:** LangSmith trace evidence from R1-05, computed at run time from `langsmith/trace_summary.csv`: drafts traced (sum of n), average seconds per draft (n-weighted mean), average cost per draft in EUR (total cost / drafts). It is part of the headline-number check. m7_human_in_loop.csv is a legacy file name; it now holds the LangSmith trace rows.

**Tableau:** the dashboard is built and published by Ayhan. The Tableau Public link, the screenshots (found at run time in `dashboard/screenshots/`), the workbook (any `.twbx` in `dashboard/`) and the 2-minute test result are all checked in Verify.

Run top to bottom. Undo only runs if `RUN_UNDO = True`.

In [1]:
# 0 Settings and 1 Check: right folder, input files exist
from pathlib import Path
from urllib.parse import quote
import subprocess, shutil, datetime, hashlib, re, json
import numpy as np
import pandas as pd

_top = subprocess.run(['git', 'rev-parse', '--show-toplevel'], capture_output=True, text=True).stdout.strip()
REPO = Path(_top) if _top else Path.cwd()
PROC = REPO / 'data' / 'processed'
DASH = REPO / 'dashboard'
DATA = DASH / 'data'
NOTEBOOK = REPO / 'notebooks' / 'R1-03_dashboard_prep.ipynb'

# inputs (the Tobi test file is deliberately NOT in this list)
IN = {
    'contacts': PROC / 'shopzilla_contacts_clean.csv',
    'email': PROC / 'shopzilla_email_clean.csv',
    'tuning': PROC / 'tobi_tuning_150.csv',
    'pulse': PROC / 'synthetic_team_pulse.csv',
}
RESEARCH = REPO / 'research' / 'sector_research.md'
# M7 input from R1-05 (read only)
TRACE = REPO / 'langsmith' / 'trace_summary.csv'
COST6 = REPO / 'cost_estimation' / 'cost_analysis.md'   # R1-06 output: touch time per email (M3)
NEED7 = ['set', 'n', 'avg_latency_s', 'total_cost_eur']

# every file this notebook creates or changes
OUT = {
    'm1': DATA / 'm1_csat_by_channel.csv',
    'm2': DATA / 'm2_email_csat_by_category.csv',
    'm3': DATA / 'm3_sla_gap.csv',
    'm4': DATA / 'm4_triage_mix.csv',
    'm5': DATA / 'm5_team_email_view.csv',
    'm6': DATA / 'm6_ai_pulse.csv',
    'm7': DATA / 'm7_human_in_loop.csv',
    'all': DATA / 'dashboard_all.csv',
    'docs': DASH / 'dashboard_documentation.md',
}
CSV_KEYS = ['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7']
RUN_UNDO = False
MIN_GROUP = 5
MIN_N = 30
URL_K = 'https://www.kaggle.com/datasets/ddosad/ecommerce-customer-service-satisfaction'
URL_H = 'https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets'
URL_E = 'https://www.edesk.com/de/blog/5-wesentliche-strategien-fuer-eine-effizientere-bearbeitung-von-kundenanfragen-im-online-handel-deutscher-markt/'
URL_N = 'https://ai.netomi.com/hubfs/Content%20Library/customerservice-benchmarkreport-ecommerce.pdf'

# the published dashboard (built by Ayhan in Tableau)
TABLEAU_URL = 'https://public.tableau.com/app/profile/ayhan.demir/viz/SpreekorbemailCSAIdraftshuman-approved/Dashboard1#2'
TEST_RESULT = 'Test run 27 Nov 2026: 1 non-technical tester, 5 of 6 passed; fixed: SYNTHETIC badge on M4 made larger'
SCREENSHOTS = sorted((DASH / 'screenshots').glob('*.png'))   # found at run time
TWBX = sorted(DASH.glob('*.twbx'))   # found at run time (the spec names dashboard.twbx, any .twbx is accepted)
results = []

def check(step, ok, note=''):
    results.append((step, 'PASS' if ok else 'FAIL', note))
    print('PASS' if ok else 'FAIL', '|', step, '|', note)

def snapshot():
    # sha256 of an explicit list of files we do not own (hash only, never .env, logs or board)
    # specs/R1-05_langsmith.md is left out: another card edits it
    files = list(PROC.glob('*.csv')) + list((REPO / 'research').glob('*.md'))
    files += [REPO / 'data' / 'README.md', REPO / 'AGENTS.md', TRACE, COST6]
    files += [p for p in (REPO / 'specs').glob('*.md') if p.name != 'R1-05_langsmith.md']
    files += [p for p in (REPO / 'notebooks').glob('*.ipynb') if p != NOTEBOOK]
    return {str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest() for p in files if p.exists()}

BEFORE = snapshot()
print('Files hashed before:', len(BEFORE))
print('Screenshots found:', [p.name for p in SCREENSHOTS] or 'none')
print('Workbooks found:', [p.name for p in TWBX] or 'none')

check('repo root found', (REPO / 'AGENTS.md').exists(), str(REPO))
for name, p in IN.items():
    check('input exists: ' + p.name, p.exists())
check('research/sector_research.md exists', RESEARCH.exists())
# the R1-05 file must exist and have the columns M7 needs
check('langsmith/trace_summary.csv exists with columns ' + ','.join(NEED7), TRACE.exists() and set(NEED7) <= set(pd.read_csv(TRACE).columns))
_c6 = COST6.read_text(encoding='utf-8') if COST6.exists() else ''
check('cost_estimation/cost_analysis.md (R1-06) exists with manual_min and ai_min rows', '| manual_min |' in _c6 and '| ai_min |' in _c6)
assert all(r[1] == 'PASS' for r in results), 'Stop: fix the FAIL above (run R1-02 and R1-05 first)'

Files hashed before: 30
Screenshots found: ['Spreekorb email CS.png']
Workbooks found: ['Spreekorb email CS_ AI drafts, human-approved.twbx']
PASS | repo root found | /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption
PASS | input exists: shopzilla_contacts_clean.csv | 
PASS | input exists: shopzilla_email_clean.csv | 
PASS | input exists: tobi_tuning_150.csv | 
PASS | input exists: synthetic_team_pulse.csv | 
PASS | research/sector_research.md exists | 
PASS | langsmith/trace_summary.csv exists with columns set,n,avg_latency_s,total_cost_eur | 
PASS | cost_estimation/cost_analysis.md (R1-06) exists with manual_min and ai_min rows | 


In [2]:
# 2 Backup: copy every output file that already exists to _backup/<timestamp>/
# OUT holds all 9 files this notebook writes (M1-M7 CSVs, dashboard_all.csv, the docs)
TS = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
BK = REPO / '_backup' / TS
existed = {}
for key, p in OUT.items():
    existed[key] = p.exists()
    if p.exists():
        dest = BK / p.relative_to(REPO)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)
print('Already existing (backed up):', [k for k, v in existed.items() if v] or 'none')
print('Backup folder:', BK)

Already existing (backed up): ['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7', 'all', 'docs']
Backup folder: /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption/_backup/20261127_150853


In [3]:
# 3a Create: seven metric CSVs (computed now from data/processed)
DATA.mkdir(parents=True, exist_ok=True)
c = pd.read_csv(IN['contacts'])
e = pd.read_csv(IN['email'])
tu = pd.read_csv(IN['tuning'])
pu = pd.read_csv(IN['pulse'])

# booleans can be read as text, so convert them the safe way
for df in (c, e):
    df['response_valid'] = df['response_valid'].astype(str).str.lower().eq('true')
    df['group_ok'] = df['group_ok'].astype(str).str.lower().eq('true')

def csat_table(df, key):
    # n and share of CSAT scores 4-5 (percent) per group; works on any pandas version
    d = df.dropna(subset=['CSAT Score'])
    return d.groupby(key)['CSAT Score'].agg(n='size', csat_4_5_pct=lambda s: round(s.between(4, 5).mean() * 100, 1)).reset_index()

# ---- M1 (A): email CSAT 4-5 share vs the other channels
m1 = csat_table(c, 'channel_name')
m1['source'] = 'Shopzilla, Kaggle ' + URL_K + ' (shopzilla_contacts_clean.csv, computed)'
m1['synthetic'] = False
share = m1.set_index('channel_name')['csat_4_5_pct']
low = share.idxmin()
inb, out_ = share['Inbound'], share['Outcall']
H1 = f'{low} customers are the least satisfied: {share[low]}% give 4-5 vs {inb}% on Inbound and {out_}% on Outcall'

# ---- M2 (A): email CSAT by category; n under 30 stays in the file with a blank value (shown = False); top 8 by volume are shown
m2 = csat_table(e, 'category')
m2['shown'] = m2['n'] >= MIN_N
k2 = int((~m2['shown']).sum())
top2 = m2[m2['shown']].sort_values('n', ascending=False).head(8)
hid2 = m2[~m2['shown']].copy()
hid2['csat_4_5_pct'] = None
m2 = pd.concat([top2, hid2]).reset_index(drop=True)
m2['source'] = 'Shopzilla, Kaggle ' + URL_K + ' (shopzilla_email_clean.csv, computed)'
m2['synthetic'] = False
low2 = top2.sort_values('csat_4_5_pct').iloc[0]
cat2, val2, n2 = low2['category'], low2['csat_4_5_pct'], int(low2['n'])
all_email = round(e['CSAT Score'].dropna().between(4, 5).mean() * 100, 1)
H2 = f'{cat2} emails have the lowest satisfaction: only {val2}% give 4-5 (n={n2}) vs {all_email}% for all email'

# ---- M3 (A): SLA gap, constants restated from research/sector_research.md (KPI baseline)
SLA = {'today_median_h': 31, 'today_within24_pct': 58, 'target_pct': 90, 'target_h': 24, 'market_expect_h': 24, 'market_avg_h': 17.3}
SRC3 = 'research/sector_research.md KPI baseline'
SRC_E = 'eDesk ' + URL_E
SRC_N = 'Netomi 2020 ' + URL_N
LAB_E = 'German shopper expectation (eDesk)'
LAB_N = 'E-commerce market average (Netomi 2020)'
# touch time per email: read from the R1-06 table rows '| manual_min | 15 | min/email | label | source |'
def r106(key):
    for ln in COST6.read_text(encoding='utf-8').split('\n'):
        cells = [x.strip() for x in ln.strip().strip('|').split('|')]
        if len(cells) >= 4 and cells[0] == key and cells[2] == 'min/email':
            return float(cells[1]), cells[3]
    raise ValueError(key + ' not found in ' + str(COST6))
man3, man3_lab = r106('manual_min')
ai3, ai3_lab = r106('ai_min')
LAB_TOUCH_MAN = man3_lab + ' (Ayhan, R1-06)'
LAB_TOUCH_AI = (ai3_lab[:-1] + ', R1-06; done second, learning bias)') if ai3_lab.endswith(')') else ai3_lab + ' (R1-06; done second, learning bias)'
SRC_TOUCH = 'cost_estimation/cost_analysis.md (R1-06 timing test)'
m3 = pd.DataFrame([
    ['Spreekorb today: median first response', SLA['today_median_h'], 'hours', 'synthetic', SRC3, True],
    ['Spreekorb today: share answered within 24 h', SLA['today_within24_pct'], 'percent', 'synthetic', SRC3, True],
    ['Target (proposed): first response within', SLA['target_h'], 'hours', 'target (proposed)', SRC3, False],
    ['Target (proposed): share of emails within 24 h', SLA['target_pct'], 'percent', 'target (proposed)', SRC3, False],
    ['Reply time German shoppers expect', SLA['market_expect_h'], 'hours', LAB_E, SRC_E, False],
    ['Average response time, e-commerce', SLA['market_avg_h'], 'hours', LAB_N, SRC_N, False],
    ['Agent touch time per email today (manual)', man3, 'minutes', LAB_TOUCH_MAN, SRC_TOUCH, False],
    ['Agent touch time per email with AI draft', ai3, 'minutes', LAB_TOUCH_AI, SRC_TOUCH, False],
], columns=['measure', 'value', 'unit', 'label', 'source', 'synthetic'])
H3 = f'SYNTHETIC: Spreekorb answers emails in a median of {SLA["today_median_h"]} h; customers expect {SLA["market_expect_h"]} h'
NOTE3 = 'Touch time is minutes, customer wait is hours: emails wait in an unprioritised queue.'

# ---- M4 (B): UC1 triage opportunity, tuning file ONLY (Tobi-Bueck is confirmed synthetic)
m4 = tu.groupby(['queue', 'language']).size().reset_index(name='tickets')
m4['share_pct'] = (m4['tickets'] / m4['tickets'].sum() * 100).round(1)
ROUTINE = ['Customer Service', 'Returns and Exchanges', 'Billing and Payments']
routine = round(tu['queue'].isin(ROUTINE).mean() * 100, 1)
m4['routine_share_pct'] = routine
m4['label'] = 'what AI triage would route'
m4['source'] = 'Tobi-Bueck, Hugging Face ' + URL_H + ' (tobi_tuning_150.csv, computed)'
m4['synthetic'] = True
m4['synthetic_status'] = 'confirmed synthetic (Ayhan, 2026-11-26)'
H4 = f'{routine}% of sample tickets are Customer Service / Returns / Billing: routine mails AI can triage and draft for human approval'

# ---- M5 (B, UC3): team-level email CSAT only (no names, no Other, group_ok rows only)
t = e[e['group_ok'] & (e['team'] != 'Other')]
m5 = csat_table(t, 'team').rename(columns={'n': 'n_emails'})
m5['shown'] = m5['n_emails'] >= MIN_N
k5 = int((~m5['shown']).sum())
m5.loc[~m5['shown'], 'csat_4_5_pct'] = None
m5 = m5.sort_values('team').reset_index(drop=True)   # alphabetical order
m5['group_ok'] = True
m5['source'] = 'Shopzilla, Kaggle ' + URL_K + ' (shopzilla_email_clean.csv, computed)'
m5['synthetic'] = False
shown5 = m5[m5['shown']]
lo5, hi5 = shown5['csat_4_5_pct'].min(), shown5['csat_4_5_pct'].max()
print('M5 computed:', lo5, hi5, len(shown5))
H5 = f'Email satisfaction ranges from {lo5}% to {hi5}% across {len(shown5)} teams: a coaching signal, not a scorecard'

# ---- M6 (C): AI adoption pulse, synthetic, overall counts, '<5' suppression (all values computed from the pulse file)
rows = [('adoption', k, v) for k, v in pu['uses_ai_answer'].value_counts().items()]
rows += [('why', k, v) for k, v in pu['why_category'].value_counts().items()]
m6 = pd.DataFrame(rows, columns=['kind', 'answer', 'count_raw'])
m6['share_pct'] = (m6['count_raw'] / len(pu) * 100).round(1)
yes_pct = m6[(m6['kind'] == 'adoption') & (m6['answer'] == 'Yes')]['share_pct'].iloc[0]
top_why = m6[m6['kind'] == 'why'].sort_values('count_raw', ascending=False).iloc[0]
top_reason, top_n = top_why['answer'], int(top_why['count_raw'])
print('M6 computed:', yes_pct, '|', top_reason, '|', top_n)
Q = chr(39)
H6 = f'{yes_pct:.0f}% would use the AI draft assistant; {Q}{top_reason}{Q} is the most common reason ({top_n} answers, synthetic)'
m6['count'] = m6['count_raw'].map(lambda n: '<5' if n < MIN_GROUP else str(n))
m6.loc[m6['count_raw'] < MIN_GROUP, 'share_pct'] = None
m6 = m6.drop(columns='count_raw')
m6['source'] = 'synthetic_team_pulse.csv (synthetic, placeholder weights)'
m6['synthetic'] = True

# ---- M7 (C): LangSmith trace evidence from R1-05, computed from langsmith/trace_summary.csv (nothing typed)
ts7 = pd.read_csv(TRACE)
n7 = int(ts7['n'].sum())                                                  # drafts traced
sec7 = round(float((ts7['n'] * ts7['avg_latency_s']).sum() / n7), 2)      # n-weighted mean seconds per draft
cost7 = round(float(ts7['total_cost_eur'].sum()) / n7, 6)                 # EUR per draft
sets7 = ' + '.join(f'{s} n={k}' for s, k in zip(ts7['set'], ts7['n']))
LAB7 = 'measured (R1-05 LangSmith, ' + sets7 + ')'
# EUR is an estimate: LangSmith gives USD from token counts; rate from langsmith/monitoring_note.md
EUR_RATE = 'USD x 0.9123, tradingview.com USDEUR, 27 Nov 2026'
LAB7_EUR = 'estimate (' + EUR_RATE + '; ' + sets7 + ')'
SRC7 = 'langsmith/trace_summary.csv + langsmith/monitoring_note.md (R1-05)'
m7 = pd.DataFrame([
    ['AI drafts traced in LangSmith', n7, 'drafts', LAB7, SRC7, False],
    ['Average seconds per draft', sec7, 'seconds', LAB7, SRC7, False],
    ['Average cost per draft', cost7, 'EUR', LAB7_EUR, SRC7, False],
], columns=['measure', 'value', 'unit', 'label', 'source', 'synthetic'])
H7 = f'{n7} held-out and timing drafts traced: {sec7:.2f} s and about EUR {cost7:.6f} per draft (gpt-4o-mini, EUR estimate)'
print('M7 computed:', n7, sec7, cost7)

# write the CSVs
for key, df in [('m1', m1), ('m2', m2), ('m3', m3), ('m4', m4), ('m5', m5), ('m6', m6), ('m7', m7)]:
    df.to_csv(OUT[key], index=False)
    print(OUT[key].name, len(df), 'rows')

# ---- dashboard_all.csv: all 7 metrics stacked with fixed columns (blank where not applicable)
COLS = ['metric', 'dim', 'series', 'value', 'unit', 'n', 'count_text', 'label', 'shown', 'highlight', 'synthetic', 'sort_order', 'source']
def stack(metric, df, **cols):
    # each argument is a column name of df, a Series, or a constant; missing ones stay blank
    out = pd.DataFrame(index=df.index)
    out['metric'] = metric
    for k in COLS[1:]:
        v = cols.get(k)
        out[k] = df[v] if isinstance(v, str) and v in df.columns else v
    out['sort_order'] = range(1, len(df) + 1)
    return out[COLS]

parts = [
    stack('M1', m1, dim='channel', series='channel_name', value='csat_4_5_pct', unit='percent 4-5', n='n', label='Email CSAT 4-5 share', shown=True, highlight=m1['channel_name'] == low, synthetic='synthetic', source='source'),
    stack('M2', m2, dim='email category', series='category', value='csat_4_5_pct', unit='percent 4-5', n='n', label='Email CSAT 4-5 share', shown='shown', highlight=m2['category'] == cat2, synthetic='synthetic', source='source'),
    stack('M3', m3, dim='SLA measure', series='measure', value='value', unit='unit', label='label', shown=True, highlight=m3['label'] == 'synthetic', synthetic='synthetic', source='source'),
    stack('M4', m4, dim='ticket queue', series='language', value='tickets', unit='ticket count', n='tickets', label='label', shown=True, highlight=m4['queue'].isin(ROUTINE), synthetic='synthetic', source='source'),
    stack('M5', m5, dim='team (pseudonym)', series='team', value='csat_4_5_pct', unit='percent 4-5', n='n_emails', label='Email CSAT 4-5 share', shown='shown', highlight=False, synthetic='synthetic', source='source'),
    stack('M6', m6, dim='kind', series='answer', value='share_pct', unit='percent', count_text='count', label='synthetic pulse', shown=True, highlight=m6['answer'] == top_reason, synthetic='synthetic', source='source'),
    stack('M7', m7, dim='trace evidence', series='measure', value='value', unit='unit', n=n7, label='label', shown=True, synthetic='synthetic', source='source'),
]
all_df = pd.concat(parts, ignore_index=True)
all_df.to_csv(OUT['all'], index=False)
print(OUT['all'].name, len(all_df), 'rows')

# metric list: tag A problem / B use case / C trust
METRICS = [
    {'id': 'M1', 'tag': 'A', 'name': 'Email CSAT vs other channels', 'headline': H1, 'verb': 'give', 'chart': 'Bar chart, one bar per channel, Email highlighted, label the percent', 'source': 'Shopzilla (Kaggle ' + URL_K + '), computed from shopzilla_contacts_clean.csv', 'synthetic': 'No (public proxy data, different company)', 'file': OUT['m1'].name},
    {'id': 'M2', 'tag': 'A', 'name': 'Email CSAT by category', 'headline': H2, 'verb': 'have', 'chart': 'Horizontal bars sorted by volume, show n in the label, filter shown = True', 'source': 'Shopzilla (Kaggle ' + URL_K + '), computed from shopzilla_email_clean.csv', 'synthetic': 'No (public proxy data)', 'file': OUT['m2'].name},
    {'id': 'M3', 'tag': 'A', 'name': 'SLA gap: today vs target vs market', 'headline': H3, 'verb': 'answers', 'chart': 'Bullet or bar chart in hours with a reference line at 24 h; colours for today, target, market; touch time (minutes) as a separate small tile', 'source': SRC3 + '; ' + SRC_E + '; ' + SRC_N + '; touch time: ' + SRC_TOUCH, 'synthetic': 'Yes for the Spreekorb-today rows (synthetic); target is target (proposed); market rows are cited (eDesk, Netomi 2020)', 'file': OUT['m3'].name},
    {'id': 'M4', 'tag': 'B', 'name': 'UC1 triage opportunity: what AI triage would route', 'headline': H4, 'verb': 'can triage', 'chart': 'Stacked bar by queue, split by language (en, de)', 'source': 'Tobi-Bueck (Hugging Face ' + URL_H + '), computed from tobi_tuning_150.csv only', 'synthetic': 'Yes (SYNTHETIC, confirmed synthetic by Ayhan, 2026-11-26)', 'file': OUT['m4'].name},
    {'id': 'M5', 'tag': 'B', 'name': 'UC3 team view: email CSAT per team', 'headline': H5, 'verb': 'ranges', 'chart': 'Dot plot of CSAT per team in alphabetical order, filter shown = True', 'source': 'Shopzilla (Kaggle ' + URL_K + '), computed from shopzilla_email_clean.csv; teams have at least 5 distinct agents per team-month (group_ok)', 'synthetic': 'No (public proxy data); team labels are pseudonyms that change on every R1-02 run', 'file': OUT['m5'].name},
    {'id': 'M6', 'tag': 'C', 'name': 'AI adoption pulse (trust)', 'headline': H6, 'verb': 'would use', 'chart': 'Two tiles: Yes/No donut plus a bar chart of reasons (why_category)', 'source': 'data/processed/synthetic_team_pulse.csv', 'synthetic': 'Yes (SYNTHETIC, made up, not Spreekorb staff)', 'file': OUT['m6'].name},
    {'id': 'M7', 'tag': 'C', 'name': 'LangSmith trace evidence (R1-05)', 'headline': H7, 'verb': 'traced', 'chart': 'Three big-number tiles: drafts traced, average seconds per draft, average cost per draft in EUR', 'source': SRC7, 'synthetic': 'No (measured latency and token cost of gpt-4o-mini calls on invented Spreekorb emails, R1-05 LangSmith held-out and timing runs; dev runs not counted; EUR cost is an estimate, ' + EUR_RATE + ')', 'file': OUT['m7'].name},
]
STORY = ['M1', 'M2', 'M3', 'M4', 'M7', 'M6', 'M5']   # M7 (trace evidence) sits after M4 and before the pulse and the team view
for m in METRICS:
    print(m['id'], m['tag'], '|', m['headline'])

M5 computed: 42.4 88.5 36
M6 computed: 56.2 | need training | 24
M7 computed: 30 2.21 0.000346
m1_csat_by_channel.csv 3 rows
m2_email_csat_by_category.csv 11 rows
m3_sla_gap.csv 8 rows
m4_triage_mix.csv 7 rows
m5_team_email_view.csv 39 rows
m6_ai_pulse.csv 8 rows
m7_human_in_loop.csv 3 rows
dashboard_all.csv 79 rows
M1 A | Email customers are the least satisfied: 73.5% give 4-5 vs 82.7% on Inbound and 83.2% on Outcall
M2 A | Cancellation emails have the lowest satisfaction: only 60.9% give 4-5 (n=64) vs 73.5% for all email
M3 A | SYNTHETIC: Spreekorb answers emails in a median of 31 h; customers expect 24 h
M4 B | 98.0% of sample tickets are Customer Service / Returns / Billing: routine mails AI can triage and draft for human approval
M5 B | Email satisfaction ranges from 42.4% to 88.5% across 36 teams: a coaching signal, not a scorecard
M6 C | 56% would use the AI draft assistant; 'need training' is the most common reason (24 answers, synthetic)
M7 C | 30 held-out and timing drafts tr

In [4]:
# 3b Create: dashboard/dashboard_documentation.md (all numbers come from the headlines above)
NL = chr(10)
TITLE = 'Email customers are our least satisfied: AI drafts, human-approved, can close the gap'
SUBTITLE = 'Spreekorb is an invented company. Tiles marked SYNTHETIC use made-up numbers. Every AI draft is approved by a person before it is sent.'
LEGACY = 'm7_human_in_loop.csv is a legacy file name; it now holds the LangSmith trace rows.'
WHY = {
    'M1': 'the problem: email customers are the least satisfied.',
    'M2': 'where it hurts inside email.',
    'M3': 'the SLA gap, clearly labelled synthetic.',
    'M4': 'what AI triage would do; a human approves every draft.',
    'M7': 'trust evidence: the held-out and timing drafts are traced (count, seconds, cost) in R1-05; EUR cost is an estimate.',
    'M6': 'trust: how the team feels about AI (synthetic).',
    'M5': 'team view, last: a coaching signal, not a scorecard.',
}
# screenshot paths are relative to dashboard/ and URL-encoded (spaces become %20)
shot_link = lambda p: 'screenshots/' + quote(p.name)
L = []
add = L.append
add('# Dashboard documentation: R1-03')
add('')
add('Title of the dashboard: ' + TITLE)
add('')
add('Subtitle: ' + SUBTITLE)
add('')
add('Tableau Public link: ' + TABLEAU_URL)
add('')
for p in TWBX:
    add('Workbook: ' + p.name)
if not TWBX:
    add('Workbook: (no .twbx found in dashboard/)')
add('')
add('Screenshots:')
for p in SCREENSHOTS:
    add('![' + p.stem + '](' + shot_link(p) + ')')
if not SCREENSHOTS:
    add('(no screenshots found in dashboard/screenshots/)')
add('')
add('Shopzilla and Tobi-Bueck are public datasets used as proxy data. The market figures in M3 are cited (eDesk, Netomi 2020), see Sources.')
add('')
add('## Sources')
add('- Shopzilla (Kaggle): ' + URL_K)
add('- Tobi-Bueck (Hugging Face): ' + URL_H)
add('- eDesk (German shopper expectation, 24 h): ' + URL_E)
add('- Netomi 2020 (e-commerce market average, 17.3 h): ' + URL_N)
add('- langsmith/trace_summary.csv (R1-05): source of M7')
add('- research/sector_research.md (KPI baseline) and research/use_cases.md')
add('')
add('## Metrics')
add('Tags: A problem, B use case, C trust. The CSV files are in dashboard/data/ and are computed from data/processed (M7: langsmith/trace_summary.csv) when the notebook R1-03_dashboard_prep runs. dashboard/data/dashboard_all.csv stacks all 7 metrics in one table (columns: ' + ', '.join(COLS) + ').')
add('')
for m in METRICS:
    add('### ' + m['id'] + ' (' + m['tag'] + ') ' + m['name'])
    add('- Headline: ' + m['headline'])
    add('- Chart type: ' + m['chart'])
    add('- Source: ' + m['source'])
    add('- Synthetic label: ' + m['synthetic'])
    add('- CSV file: dashboard/data/' + m['file'])
    add('')
add('## Notes on hidden values and labels')
add(f'- M2: {k2} categories hidden (n<{MIN_N}). Only the top 8 categories by volume (n of at least {MIN_N}) are shown. Hidden rows stay in the CSV with n and a blank value (shown = False).')
add(f'- M5: {k5} teams hidden (n<{MIN_N}). Hidden rows stay in the CSV with n and a blank value (shown = False).')
add('- M5: the minimum of 5 distinct agents per team-month was asserted in R1-02 before names were dropped (flag group_ok). This card relies on that flag. No names are shown.')
add('- M5: a team view needs works council (Betriebsrat) agreement and a DPIA before real use (see data/README.md).')
add('- M3: ' + NOTE3 + f' Touch time per email: {man3:g} min manual ({LAB_TOUCH_MAN}) and {ai3:g} min with the AI draft ({LAB_TOUCH_AI}), from ' + SRC_TOUCH + '.')
add('- M6: counts under 5 are shown as <5.')
add('- M7: computed from langsmith/trace_summary.csv: drafts traced = sum of n, seconds per draft = n-weighted mean of avg_latency_s, cost per draft = total cost in EUR / drafts. ' + LEGACY)
add('')
add('## Story order: Chleo understands in 2 minutes')
for i, mid in enumerate(STORY, 1):
    tag = [m['tag'] for m in METRICS if m['id'] == mid][0]
    add(str(i) + '. ' + mid + ' (' + tag + '): ' + WHY[mid])
add('')
add('## Tableau build steps (beginner-friendly)')
add('1. Open Tableau Public. Choose Connect > Text file and open one CSV from dashboard/data/.')
add('2. Make one sheet per metric (M1 to M7). Drag the fields to Columns and Rows and pick the chart type above. For M2 and M5 filter shown = True.')
add('3. Put the headline of each metric into the sheet title. Use the text exactly as written above.')
add('4. Create a new Dashboard. Drag the sheets in story order. Use a wide layout (for example 1200 x 900).')
add('5. Put a SYNTHETIC badge on M3, M4, M6 and a source footer on every tile (source text and CSV name).')
add('6. Label the M3 target as target (proposed) and the market values with their source (eDesk, Netomi 2020).')
add('7. Add the title and the subtitle above as text boxes at the top.')
add('8. Save the packaged workbook as a .twbx in dashboard/ (File > Save As, packaged workbook).')
add('9. Publish: File > Save to Tableau Public (done, link above).')
add('10. Take screenshots into dashboard/screenshots/ (done, embedded above).')
add('')
add('## 2-minute test (Ayhan ticks)')
add('Ask Chleo (or a colleague playing Chleo) to look at the dashboard for 2 minutes without help.')
add('- [ ] She can say what the problem is (email customers are the least satisfied, first response is slow).')
add('- [ ] She can say why it is worth solving (the gap to the target and the market).')
add('- [ ] She sees where each number comes from (source footer on every tile).')
add('- [ ] She can tell which tiles are synthetic (M3, M4, M6).')
add('- [ ] She sees that a person approves every AI draft (see the subtitle and M4).')
add('- [ ] The team tile reads as a coaching signal, not a scorecard.')
add('')
add('Result: ' + TEST_RESULT)
add('')
OUT['docs'].write_text(NL.join(L), encoding='utf-8')
print('Docs written:', OUT['docs'], '|', len(L), 'lines')

Docs written: /Users/macbook-air/Desktop/AI consulting/Week8/Day4/ecommerce_customersatisfaction_adiadoption/dashboard/dashboard_documentation.md | 116 lines


In [5]:
# 4 Verify: one PASS/FAIL per acceptance item
n_before = len(results)
read = {k: pd.read_csv(OUT[k]) for k in CSV_KEYS}
ra = pd.read_csv(OUT['all'])
docs = OUT['docs'].read_text(encoding='utf-8')
sector = RESEARCH.read_text(encoding='utf-8')
is_true = lambda s: s.fillna('').astype(str).str.lower().eq('true')
NUM = r'\d+(?:\.\d+)?'
nums = lambda text: {float(x) for x in re.findall(NUM, text)}   # compare numbers as floats (56 equals 56.0)

# spec 1: 5-7 metrics, each tagged A/B/C
check('5-7 metrics, each tagged A/B/C', 5 <= len(METRICS) <= 7 and all(m['tag'] in ('A', 'B', 'C') for m in METRICS), str(len(METRICS)) + ' metrics, tags ' + ''.join(m['tag'] for m in METRICS))
check('all three tags A, B and C are used', {m['tag'] for m in METRICS} == {'A', 'B', 'C'})

# spec 2: headlines are insights: a verb, a number, and a number from the metric's own value / n columns (M7 included now)
bad = [m['id'] for m in METRICS if not (m['verb'] in m['headline'] and re.search(r'\d', m['headline']))]
check('each headline has a verb and a number', not bad, 'failed: ' + str(bad) if bad else 'ok')
# value and n columns only, so source URLs never count (M4: share column, M6: count column are that metric's value columns)
NUMCOLS = {'M1': ['csat_4_5_pct', 'n'], 'M2': ['csat_4_5_pct', 'n'], 'M3': ['value'], 'M4': ['tickets', 'routine_share_pct'], 'M5': ['csat_4_5_pct', 'n_emails'], 'M6': ['share_pct', 'count'], 'M7': ['value']}
bad = []
for m in METRICS:
    head_nums = nums(m['headline'].replace('4-5', '').replace('gpt-4o-mini', ''))
    d = read[m['id'].lower()]
    raw = nums(' '.join(' '.join(d[col].fillna('').astype(str)) for col in NUMCOLS[m['id']]))
    data_nums = raw | {float(round(x)) for x in raw}   # rounded values count, e.g. 56 for 56.2
    if not head_nums & data_nums:
        bad.append(m['id'])
check('each headline has a number that appears in its own value / n columns (M7 included)', not bad, 'failed: ' + str(bad) if bad else 'ok')
check('headlines are in the docs', all(m['headline'] in docs for m in METRICS))
BANNED = ['blame', 'worst', 'best team', 'bottom', 'top performer', 'ranking']
all_text = (' '.join(m['headline'] for m in METRICS) + ' ' + docs).lower()
hit = [w for w in BANNED if w in all_text]
check('no banned words in headlines and docs', not hit, str(hit) if hit else 'ok')

# the email CSAT share is read from the research file, tolerance 0.2
found = re.search(r'lowest share of CSAT scores 4-5 at (' + NUM + r')%', sector)
doc_val = float(found.group(1)) if found else None
check('email is the lowest CSAT channel and matches research (+-0.2)', low == 'Email' and doc_val is not None and abs(share['Email'] - doc_val) <= 0.2, 'computed ' + str(share['Email']) + ' vs research ' + str(doc_val))
check('SLA constants match research/sector_research.md', all(s in sector for s in ['median 31 h', '58% within 24 h', '17.3 h', '90%']))

# spec 3: sources and synthetic labels in every CSV
check('each CSV has source and synthetic columns', all({'source', 'synthetic'} <= set(df.columns) for df in read.values()))
syn_ok = (is_true(read['m3'].loc[read['m3']['label'] == 'synthetic', 'synthetic']).all() and not is_true(read['m3'].loc[read['m3']['label'] != 'synthetic', 'synthetic']).any()
          and is_true(read['m4']['synthetic']).all() and is_true(read['m6']['synthetic']).all()
          and not any(is_true(read[k]['synthetic']).any() for k in ['m1', 'm2', 'm5', 'm7']))
check('synthetic=True for M3 today rows, M4 and M6; False elsewhere', bool(syn_ok))
check('M4 synthetic_status says confirmed and docs have no to-confirm wording for Tobi', read['m4']['synthetic_status'].str.startswith('confirmed synthetic').all() and 'to confirm' not in docs.lower())
check('docs name a source and synthetic label for every metric', docs.count('- Source: ') == len(METRICS) and docs.count('- Synthetic label: ') == len(METRICS))
check('Kaggle and Hugging Face URLs are in the docs and in the source columns', URL_K in docs and URL_H in docs and read['m1']['source'].str.contains(URL_K, regex=False).all() and read['m4']['source'].str.contains(URL_H, regex=False).all())

# Tobi test file never read; tuning file only
all_csv_text = ' '.join(OUT[k].read_text(encoding='utf-8') for k in CSV_KEYS + ['all'])
check('no CSV reads the Tobi test file', 'tobi_test' not in all_csv_text and all('test' not in p.name for p in IN.values()), 'inputs: ' + ', '.join(p.name for p in IN.values()))
check('tuning file split column is all tuning, M4 has 150 tickets', set(tu['split']) == {'tuning'} and read['m4']['tickets'].sum() == len(tu) == 150, 'M4 tickets = ' + str(read['m4']['tickets'].sum()))

# team metric: group_ok only, no names, no Other, no highest/lowest wording
m5r = read['m5']
check('team metric: group_ok all True, no Other, labels are Team NN only', len(m5r) >= MIN_GROUP and is_true(m5r['group_ok']).all() and m5r['team'].str.fullmatch(r'Team \d\d').all(), str(len(m5r)) + ' teams in file')
check('team metric has no name columns', not any(w in col.lower() for col in m5r.columns for w in ['name', 'agent', 'supervisor', 'manager']))
check('team headline has no team label and no highest/lowest wording', not re.search(r'Team \d', H5) and not any(w in H5.lower() for w in ['highest', 'lowest']))
check('M5 is tagged B (UC3)', [m['tag'] for m in METRICS if m['id'] == 'M5'] == ['B'])

# suppression: M2 and M5 hidden rows keep n, blank value, shown False; shown rows have n >= 30
sup_ok = True
for k, ncol in [('m2', 'n'), ('m5', 'n_emails')]:
    d = read[k]
    sh = is_true(d['shown'])
    sup_ok = sup_ok and (d.loc[sh, ncol] >= MIN_N).all() and d.loc[~sh, 'csat_4_5_pct'].isna().all() and (d.loc[~sh, ncol] < MIN_N).all() and d[ncol].notna().all()
check('M2 and M5: hidden rows keep n, blank value, shown=False', bool(sup_ok), 'hidden: ' + str(k2) + ' categories, ' + str(k5) + ' teams')
check('docs state the hidden counts', f'{k2} categories hidden (n<{MIN_N})' in docs and f'{k5} teams hidden (n<{MIN_N})' in docs)
check('pulse counts under 5 are suppressed', all(v == '<5' or int(v) >= MIN_GROUP for v in read['m6']['count'].dropna().astype(str)))

# spec 4: docs structure and story
check('docs have story order, build steps and 2-minute checklist', all(s in docs for s in ['## Story order', '## Tableau build steps', '## 2-minute test', '- [ ]']))
check('story order is M1, M2, M3, M4, M7, M6, M5 in the docs, and the docs list M7 as built (no pending wording)', STORY == ['M1', 'M2', 'M3', 'M4', 'M7', 'M6', 'M5'] and docs.index('1. M1') < docs.index('2. M2') < docs.index('3. M3') < docs.index('4. M4') < docs.index('5. M7') < docs.index('6. M6') < docs.index('7. M5') and 'M7 added after' not in docs)
check('docs say SYNTHETIC badge on M3, M4, M6 and a source footer', 'SYNTHETIC badge on M3, M4, M6 and a source footer on every tile' in docs)

# fix round F1 to F4 (one check each)
m3r = read['m3']
touch = m3r[m3r['unit'] == 'minutes']
check('F1: M5 has no response column; M3 has 2 touch-time rows filled from R1-06 (manual estimate, AI measured, not synthetic); docs have the M3 note',
      not any('response' in col.lower() for col in m5r.columns) and len(touch) == 2 and touch['value'].tolist() == [r106('manual_min')[0], r106('ai_min')[0]]
      and touch['label'].iloc[0].startswith('estimate') and touch['label'].iloc[1].startswith('measured') and (touch['source'] == SRC_TOUCH).all() and not is_true(touch['synthetic']).any()
      and 'pending r1-06' not in docs.lower()
      and NOTE3 in docs and 'median_response' not in docs and 'median response hours' not in docs.lower())
lab24 = m3r.loc[m3r['value'] == 24.0, 'label'].tolist()
lab17 = m3r.loc[m3r['value'] == 17.3, 'label'].tolist()
check('F2: both market labels are present', LAB_E in lab24 and LAB_N in lab17, str(lab24) + ' | ' + str(lab17))
yes_exp = round((pu['uses_ai_answer'] == 'Yes').mean() * 100, 1)
vc = pu['why_category'].value_counts()
exp_head = f'{yes_exp:.0f}% would use the AI draft assistant; {chr(39)}{vc.index[0]}{chr(39)} is the most common reason ({int(vc.iloc[0])} answers, synthetic)'
nb = json.loads(NOTEBOOK.read_text(encoding='utf-8'))
src_of = lambda cell: ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
h6_lines = [ln for cell in nb['cells'] if cell['cell_type'] == 'code' for ln in src_of(cell).split(chr(10)) if ln.startswith(('H6 =', 'yes_pct =', 'top_why ='))]
check('F3: M6 headline matches the values computed from the pulse; no 56 or 24 typed in its code', H6 == exp_head and len(h6_lines) == 3 and not any(re.search(r'\b(56|24)\b', ln) for ln in h6_lines), H6)
check('F4: eDesk and Netomi URLs in the M3 CSV and docs; no link-to-add text in any output', URL_E in m3r['source'].str.cat() and URL_N in m3r['source'].str.cat() and URL_E in docs and URL_N in docs and 'link to add' not in (all_csv_text + docs).lower() and 'links to be added' not in (all_csv_text + docs).lower())

# update round: M7 (now filled from R1-05), subtitle, H3, H5
m7r = read['m7']
want7 = {'AI drafts traced in LangSmith': 'drafts', 'Average seconds per draft': 'seconds', 'Average cost per draft': 'EUR'}
# recompute from the R1-05 file a second way (np.average) and compare
tsv = pd.read_csv(TRACE)
n_chk = int(tsv['n'].sum())
sec_chk = round(float(np.average(tsv['avg_latency_s'], weights=tsv['n'])), 2)
cost_chk = round(float(tsv['total_cost_eur'].sum()) / n_chk, 6)
check('M7: 3 rows, right measures and units, values not blank, label and source, synthetic False, tag C',
      len(m7r) == 3 and dict(zip(m7r['measure'], m7r['unit'])) == want7 and m7r['value'].notna().all() and m7r['label'].iloc[:2].str.startswith('measured (R1-05 LangSmith').all() and m7r['label'].iloc[2].startswith('estimate (USD x 0.9123') and (m7r['source'] == SRC7).all()
      and not is_true(m7r['synthetic']).any() and [m['tag'] for m in METRICS if m['id'] == 'M7'] == ['C'])
got7 = dict(zip(m7r['measure'], m7r['value']))
check('M7 values equal a recomputation from langsmith/trace_summary.csv', np.allclose([got7['AI drafts traced in LangSmith'], got7['Average seconds per draft'], got7['Average cost per draft']], [n_chk, sec_chk, cost_chk], rtol=0, atol=1e-9), str(n_chk) + ' drafts, ' + str(sec_chk) + ' s, EUR ' + str(cost_chk))
check('M7 headline is built from the recomputed values', H7 == f'{n_chk} held-out and timing drafts traced: {sec_chk:.2f} s and about EUR {cost_chk:.6f} per draft (gpt-4o-mini, EUR estimate)', H7)
_r = pd.read_csv(TRACE)
check('M7 EUR rate in trace_summary.csv equals the cited 0.9123 (+-0.001) and the rate is in the docs', bool(((_r['total_cost_eur'] / _r['total_cost_usd']) - 0.9123).abs().max() < 0.001) and EUR_RATE in docs, EUR_RATE)
check('no "pending R1-05" left in dashboard_documentation.md', 'pending r1-05' not in docs.lower())
check('subtitle is in the docs', SUBTITLE in docs and SUBTITLE == 'Spreekorb is an invented company. Tiles marked SYNTHETIC use made-up numbers. Every AI draft is approved by a person before it is sent.')
check('docs say m7_human_in_loop.csv is a legacy file name', LEGACY == 'm7_human_in_loop.csv is a legacy file name; it now holds the LangSmith trace rows.' and LEGACY in docs and OUT['m7'].name == 'm7_human_in_loop.csv')
item = [ln for ln in docs.split(chr(10)) if 'approves every AI draft' in ln]
check('2-minute item about the human approval mentions both the subtitle and M4', len(item) == 1 and 'subtitle' in item[0] and 'M4' in item[0], item[0] if item else 'line not found')

# NOTE (not a check): which M1-M6 CSVs changed vs the backup.
# M1-M6 are recomputed from data/processed, so they change when R1-02 was re-run
# (27 Nov e-commerce filter fix changed M4 and the M5 team numbers). Update those tiles in Tableau too.
diff6 = [k for k in CSV_KEYS[:6] if existed[k] and (BK / OUT[k].relative_to(REPO)).read_bytes() != OUT[k].read_bytes()]
print('NOTE | M1-M6 CSVs changed vs backup (recomputed from data/processed):', diff6 if diff6 else 'none')

# H3 is checked against the values read back from the M3 CSV (not against the constants)
today = m3r[(m3r['label'] == 'synthetic') & (m3r['unit'] == 'hours')]
expect = m3r[m3r['label'] == LAB_E]
today_syn = m3r[m3r['label'] == 'synthetic']
h3_ok = len(today) == 1 and len(expect) == 1 and ('median of ' + format(float(today['value'].iloc[0]), 'g') + ' h') in H3 and ('expect ' + format(float(expect['value'].iloc[0]), 'g') + ' h') in H3
check('H3 contains the synthetic today-median and the 24 h expectation read from the M3 CSV', bool(h3_ok), H3)
check('M3 Spreekorb-today rows are synthetic=True in the CSV and H3 starts with SYNTHETIC:', len(today_syn) == 2 and is_true(today_syn['synthetic']).all() and H3.startswith('SYNTHETIC:'))
sh5 = read['m5'][is_true(read['m5']['shown'])]
exp5 = 'Email satisfaction ranges from ' + str(sh5['csat_4_5_pct'].min()) + '% to ' + str(sh5['csat_4_5_pct'].max()) + '% across ' + str(len(sh5)) + ' teams: a coaching signal, not a scorecard'
check('H5 matches the format and the shown M5 rows', H5 == exp5, H5)
check('H5 has no en dash or em dash', chr(8211) not in H5 and chr(8212) not in H5)

# dashboard_all.csv against the 7 CSVs: value, n, unit, label, shown, synthetic, source and sort_order per metric
norm = lambda s: s.fillna('').astype(str).str.lower().reset_index(drop=True)   # NaN becomes empty, True/False become true/false
numc = lambda s: pd.to_numeric(s, errors='coerce').astype(float).values
def want_text(df, spec):
    if spec is None:
        return pd.Series([''] * len(df))
    if isinstance(spec, str) and spec in df.columns:
        return norm(df[spec])
    return pd.Series([str(spec).lower()] * len(df))
EXP = {
    'M1': ('m1', dict(value='csat_4_5_pct', n='n', unit='percent 4-5', label='Email CSAT 4-5 share', shown=True)),
    'M2': ('m2', dict(value='csat_4_5_pct', n='n', unit='percent 4-5', label='Email CSAT 4-5 share', shown='shown')),
    'M3': ('m3', dict(value='value', n=None, unit='unit', label='label', shown=True)),
    'M4': ('m4', dict(value='tickets', n='tickets', unit='ticket count', label='label', shown=True)),
    'M5': ('m5', dict(value='csat_4_5_pct', n='n_emails', unit='percent 4-5', label='Email CSAT 4-5 share', shown='shown')),
    'M6': ('m6', dict(value='share_pct', n=None, unit='percent', label='synthetic pulse', shown=True)),
    'M7': ('m7', dict(value='value', n=n_chk, unit='unit', label='label', shown=True)),   # M7 n = sum(n) from trace_summary.csv
}
all_ok = list(ra.columns) == COLS and len(ra) == sum(len(df) for df in read.values()) and set(ra['metric']) == {'M' + str(i) for i in range(1, 8)}
bad_cols = []
for mid, (k, spec) in EXP.items():
    df, sub = read[k], ra[ra['metric'] == mid].reset_index(drop=True)
    if len(sub) != len(df):
        bad_cols.append(mid + ' rows')
        continue
    for col in ['value', 'n']:
        s = spec[col]
        if s is None:
            want = np.full(len(df), np.nan)
        elif isinstance(s, str):
            want = numc(df[s])
        else:
            want = np.full(len(df), float(s))   # a constant, e.g. M7 n
        if not np.allclose(numc(sub[col]), want, equal_nan=True):
            bad_cols.append(mid + '.' + col)
    if not (numc(sub['sort_order']) == np.arange(1, len(df) + 1)).all():
        bad_cols.append(mid + '.sort_order')
    for col, s in [('unit', spec['unit']), ('label', spec['label']), ('shown', spec['shown']), ('synthetic', 'synthetic'), ('source', 'source')]:
        if not (norm(sub[col]) == want_text(df, s)).all():
            bad_cols.append(mid + '.' + col)
check('dashboard_all.csv: exact columns, row count, metrics M1-M7, and value, n, unit, label, shown, synthetic, source, sort_order match the 7 CSVs', bool(all_ok) and not bad_cols, str(len(ra)) + ' rows' + (' | mismatch: ' + str(bad_cols) if bad_cols else ''))

# Tableau deliverables built by Ayhan: link, screenshots, test result, workbook, no placeholders
check('(a) Tableau Public link is in the docs and starts with https://public.tableau.com/', ('Tableau Public link: ' + TABLEAU_URL) in docs and TABLEAU_URL.startswith('https://public.tableau.com/'), TABLEAU_URL)
shot_missing = [p.name for p in SCREENSHOTS if shot_link(p) not in docs]
check('(b) at least 1 png in dashboard/screenshots/ and each png is referenced in the docs', len(SCREENSHOTS) >= 1 and not shot_missing, str(len(SCREENSHOTS)) + ' png found' + (' | not referenced: ' + str(shot_missing) if shot_missing else ''))
check('(c) the docs contain 5 of 6 passed (2-minute test result)', '5 of 6 passed' in docs and ('Result: ' + TEST_RESULT) in docs, TEST_RESULT)
tw_missing = [p.name for p in TWBX if ('Workbook: ' + p.name) not in docs]
check('(d) at least 1 .twbx exists in dashboard/ and the docs name it (the spec names dashboard.twbx, any .twbx is accepted)', len(TWBX) >= 1 and not tw_missing, ', '.join(p.name for p in TWBX) if TWBX else 'no .twbx found: save the packaged workbook in dashboard/')
check('(e) no TO ADD left in the docs', 'TO ADD' not in docs)

mine = results[n_before:]
fails = [r for r in mine if r[1] == 'FAIL']
print('\nVERIFY SUMMARY:', len(mine) - len(fails), 'PASS,', len(fails), 'FAIL')
print('OVERALL (all checks, Tableau items included, nothing pending):', 'PASS' if not fails else 'FAIL')

PASS | 5-7 metrics, each tagged A/B/C | 7 metrics, tags AAABBCC
PASS | all three tags A, B and C are used | 
PASS | each headline has a verb and a number | ok
PASS | each headline has a number that appears in its own value / n columns (M7 included) | ok
PASS | headlines are in the docs | 
PASS | no banned words in headlines and docs | ok
PASS | email is the lowest CSAT channel and matches research (+-0.2) | computed 73.5 vs research 73.5
PASS | SLA constants match research/sector_research.md | 
PASS | each CSV has source and synthetic columns | 
PASS | synthetic=True for M3 today rows, M4 and M6; False elsewhere | 
PASS | M4 synthetic_status says confirmed and docs have no to-confirm wording for Tobi | 
PASS | docs name a source and synthetic label for every metric | 
PASS | Kaggle and Hugging Face URLs are in the docs and in the source columns | 
PASS | no CSV reads the Tobi test file | inputs: shopzilla_contacts_clean.csv, shopzilla_email_clean.csv, tobi_tuning_150.csv, synthetic_tea

In [6]:
# 5 Unchanged: files we do not own keep the same hash (data/processed, research, README, AGENTS, specs, other notebooks, trace_summary.csv)
AFTER = snapshot()
check('hashed some files before', len(BEFORE) > 0, str(len(BEFORE)) + ' files')
changed = sorted(k for k in BEFORE if AFTER.get(k) != BEFORE[k])   # a missing file counts as changed
check('data/processed/* unchanged', not any(k.startswith('data/processed') for k in changed))
check('research/*.md, data/README.md, AGENTS.md, specs/*.md unchanged', not any(k.startswith(('research', 'specs', 'AGENTS', 'data/README')) for k in changed))
check('langsmith/trace_summary.csv unchanged (read only)', 'langsmith/trace_summary.csv' in BEFORE and 'langsmith/trace_summary.csv' not in changed)
check('cost_estimation/cost_analysis.md unchanged (read only)', 'cost_estimation/cost_analysis.md' in BEFORE and 'cost_estimation/cost_analysis.md' not in changed)
check('no other existing file changed', changed == [], str(changed) if changed else 'ok, ' + str(len(AFTER)) + ' files compared')

PASS | hashed some files before | 30 files
PASS | data/processed/* unchanged | 
PASS | research/*.md, data/README.md, AGENTS.md, specs/*.md unchanged | 
PASS | langsmith/trace_summary.csv unchanged (read only) | 
PASS | cost_estimation/cost_analysis.md unchanged (read only) | 
PASS | no other existing file changed | ok, 30 files compared


In [ ]:
# 6 Undo (only if RUN_UNDO = True): restore backups, delete files that did not exist before
if RUN_UNDO:
    for key, p in OUT.items():
        if existed[key]:
            shutil.copy2(BK / p.relative_to(REPO), p)
        else:
            p.unlink(missing_ok=True)
    check('undo', all(p.exists() == existed[k] for k, p in OUT.items()), 'outputs restored or removed')
else:
    print('Undo skipped (RUN_UNDO = False)')

In [7]:
# 7 Git: print only, never run. Ayhan commits.
print('git status')
print('git add dashboard/data dashboard/dashboard_documentation.md dashboard/*.twbx dashboard/screenshots notebooks/R1-03_dashboard_prep.ipynb')
print('git commit -m "R1-03 fill M7 from R1-05 trace summary"')
print('git push')

git status
git add dashboard/data dashboard/dashboard_documentation.md dashboard/*.twbx dashboard/screenshots notebooks/R1-03_dashboard_prep.ipynb
git commit -m "R1-03 fill M7 from R1-05 trace summary"
git push
